# Notebook 3: Import / Generate Your Data

In this notebook, we demonstrate several approaches for importing and generating data for hyperscanning analysis:

- **MATLAB Data:** Load data from a MATLAB file (e.g. .mat file) using `scipy.io.loadmat`.
- **EEG Data:** Import EEG data using MNE (from a .fif file).
- **fNIRS Data:** Import fNIRS data (e.g., from a NIRx system) using MNE.
- **Simulation:** Generate synthetic data using MNE’s source simulation tools.

In [ ]:
import os
import numpy as np
import mne
from scipy.io import loadmat
import matplotlib.pyplot as plt

from hypyp.fnirs_tools import load_fnirs
from hypyp.fnirs_tools import make_fnirs_montage
from hypyp.fnirs_tools import fnirs_epoch

from mne import Epochs, compute_covariance, find_events, make_ad_hoc_cov
from mne.datasets import sample
from mne.simulation import (
    add_ecg,
    add_eog,
    add_noise,
    simulate_raw,
    simulate_sparse_stc,
)

print("Libraries imported successfully.")

## 1. Importing MATLAB Data

Here we load MATLAB data from a .mat file using `scipy.io.loadmat`.
Assume that `data_matlab.mat` contains a variable called 'data' (for example, EEG signals or similar).

In [ ]:
# Path to your MATLAB file
filename = "pce01_P1_Rest1"
matlab_rest_file = os.path.join("./data/pce01230807/eeg_recordings/", f"{filename}.mat")

# Load the .mat file
try:
    mat_rest = loadmat(matlab_rest_file)
    print(f"Loaded trial file: {matlab_rest_file}. Available keys: {list(mat_rest.keys())}")

    rest_key = "pce_P1_Rest"
    data_matlab = mat_rest.get(rest_key)
    if data_matlab is None:
        raise KeyError(f"Variable {rest_key} not found in the MATLAB file.")
    print("MATLAB data loaded successfully. Data shape:", data_matlab.shape)
except Exception as e:
    print("Error loading MATLAB data:", e)

sfreq_rest = 1000
print(f"Resting state data shape: {data_matlab.shape}, SFreq: {sfreq_rest} Hz")

n_channels_rest, n_samples_rest = data_matlab.shape
ch_names = [f'EEG{i+1}' for i in range(n_channels_rest)]
data_rest = data_matlab.astype(np.float32)
info_rest = mne.create_info(ch_names=ch_names, sfreq=sfreq_rest, ch_types=['eeg'] * n_channels_rest)
raw_rest = mne.io.RawArray(data_rest, info_rest)

try:
    montage_std = mne.channels.make_standard_montage('standard_1020')
    raw_rest.set_montage(montage_std, on_missing='ignore')
    print("Standard montage applied to resting state as fallback.")
except Exception as e:
    print("Error setting standard montage for resting state:", e)


fname = os.path.join("./data/pce01230807/", f"{filename}_raw.fif")
raw_rest.save(fname, overwrite=True)

## 2. Importing EEG Data using MNE

We load EEG data from a .fif file using MNE.  
Make sure your file (e.g., `eeg_data.fif`) contains raw EEG recordings.

In [ ]:
# Path to your EEG .fif file
filename = "pce01_P1_Rest1"
eeg_file = os.path.join("./data/pce01230807/", f"{filename}_raw.fif")

try:
    raw_eeg = mne.io.read_raw_fif(eeg_file, preload=True)
    # Optionally, set a montage if not already set (e.g., standard 10-20 montage)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw_eeg.set_montage(montage)
    print("EEG data loaded successfully. Info:")
    print(raw_eeg.info)
    
    # Plot a short segment of the raw data
    raw_eeg.plot(n_channels=8, duration=5, title="Raw EEG Data")
except Exception as e:
    print("Error loading EEG data:", e)

## 3. Importing fNIRS Data

MNE provides readers for fNIRS data, such as data from NIRx systems.
Below, we assume that the fNIRS data is stored in a folder (e.g., `nirx_data/`) as exported from a NIRx device.

In [ ]:
# Define file paths for the SNIRF data for two participants
path_1 = "./data/FNIRS/DCARE_02_sub1.snirf"
path_2 = "./data/FNIRS/DCARE_02_sub2.snirf"

print('Data paths set:')
print('Participant 1:', path_1)
print('Participant 2:', path_2)

In [ ]:
# Load fNIRS data for two participants
fnirs_data = load_fnirs(path_1, path_2, attr=None, preload=False, verbose=None)
fnirs_participant_1 = fnirs_data[0]
fnirs_participant_2 = fnirs_data[1]

print('fNIRS data loaded for both participants.')

In [ ]:
# Define source and detector labels
source_labels = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8']
detector_labels = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8']

# Path to the probe information file
prob_mat_file = './data/FNIRS/MCARE_01_probeInfo.mat'

# 3D coordinates (in mm) of anatomical landmarks
Nz_coord = [12.62, 17.33, 16.74]    # Tip of the nose
RPA = [21.0121020904262, 15.9632489747085, 17.2796094659563]    # Right preauricular
LPA = [4.55522116441745, 14.6744377188919, 18.3544292678269]     # Left preauricular

# Head size in mm
head_size = 0.16

# Create the montage using the provided probe information
location = make_fnirs_montage(source_labels, detector_labels, prob_mat_file,
                              Nz_coord, RPA, LPA, head_size)

print('Compatible montage created.')

In [ ]:
# Create Epoch objects for both participants
fnirs_epochs = fnirs_epoch(fnirs_participant_1, fnirs_participant_2,
                           tmin=-0.1, tmax=1, baseline=(None, 0),
                           preload=True, event_repeated='merge')

fnirs_epo1 = fnirs_epochs[0]
fnirs_epo2 = fnirs_epochs[1]

print('Epoch objects created for both participants.')

In [ ]:
# Set the montage for both epoch objects
fnirs_epo1.set_montage(location)
fnirs_epo2.set_montage(location)

print('Montage applied to both participants.')

In [ ]:
# Plot sensor locations with channel names for participant 1
fnirs_epo1.plot_sensors(show_names=True)

# Plot sensor locations with channel names for participant 2
fnirs_epo2.plot_sensors(show_names=True)

print('Sensor plots displayed.')

In [ ]:
# Plot the fNIRS data for participant 1
fnirs_epo1.plot()

# Plot the fNIRS data for participant 2
fnirs_epo2.plot()

print('fNIRS data plots displayed.')

## 4. Simulating Data using MNE's Source Simulation

For simulation, we use MNE’s source simulation tools. This example creates a sparse source estimate from a sample subject’s source space.  
It then displays the simulated source-level data. This approach (from the [source_simulator example](https://mne.tools/1.8/auto_examples/simulation/source_simulator.html)) is useful for testing pipelines.

In [ ]:
meg_path = os.path.join("./data/", "MNE-sample-data", "MEG", "sample")
raw_fname = os.path.join(meg_path, "sample_audvis_raw.fif")
fwd_fname = os.path.join(meg_path, "sample_audvis-meg-eeg-oct-6-fwd.fif")

# Load real data as the template
raw = mne.io.read_raw_fif(raw_fname)
raw.set_eeg_reference(projection=True)

In [ ]:
n_dipoles = 4  # number of dipoles to create
epoch_duration = 2.0  # duration of each epoch/event
n = 0  # harmonic number
rng = np.random.RandomState(0)  # random state (make reproducible)

def data_fun(times):
    """Generate time-staggered sinusoids at harmonics of 10Hz."""
    global n
    n_samp = len(times)
    window = np.zeros(n_samp)
    start, stop = (
        int(ii * float(n_samp) / (2 * n_dipoles)) for ii in (2 * n, 2 * n + 1)
    )
    window[start:stop] = 1.0
    n += 1
    data = 25e-9 * np.sin(2.0 * np.pi * 10.0 * n * times)
    data *= window
    return data


times = raw.times[: int(raw.info["sfreq"] * epoch_duration)]
fwd = mne.read_forward_solution(fwd_fname)
src = fwd["src"]
stc = simulate_sparse_stc(
    src, n_dipoles=n_dipoles, times=times, data_fun=data_fun, random_state=rng
)
# look at our source data
fig, ax = plt.subplots(1)
ax.plot(times, 1e9 * stc.data.T)
ax.set(ylabel="Amplitude (nAm)", xlabel="Time (s)")
mne.viz.utils.plt_show()

In [ ]:
raw_sim = simulate_raw(raw.info, [stc] * 10, forward=fwd, verbose=True)
cov = make_ad_hoc_cov(raw_sim.info)
add_noise(raw_sim, cov, iir_filter=[0.2, -0.2, 0.04], random_state=rng)
add_ecg(raw_sim, random_state=rng)
add_eog(raw_sim, random_state=rng)
raw_sim.plot()

In [ ]:
events = find_events(raw_sim)  # only 1 pos, so event number == 1
epochs = Epochs(raw_sim, events, 1, tmin=-0.2, tmax=epoch_duration)
cov = compute_covariance(
    epochs, tmax=0.0, method="empirical", verbose="error"
)  # quick calc
evoked = epochs.average()
evoked.plot_white(cov, time_unit="s")

## Summary

In this notebook, we have demonstrated multiple methods to import and generate data for hyperscanning analyses:

- **MATLAB Data:** Loaded using `scipy.io.loadmat`.
- **EEG Data:** Loaded from a .fif file using MNE.
- **fNIRS Data:** Imported from a NIRx folder using MNE.
- **Simulation:** Generated simulated source-level data using MNE's `simulate_sparse_stc`.

These techniques provide a solid foundation for subsequent analyses in your hyperscanning workflow.